In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Sampler 옵션 지정

<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
옵션을 사용하여 Sampler 프리미티브를 커스터마이즈할 수 있습니다. 이 섹션에서는 Qiskit Runtime 프리미티브 옵션을 지정하는 방법에 초점을 맞춥니다. 프리미티브의 `run()` 메서드 인터페이스는 모든 구현에서 공통이지만, 옵션은 그렇지 않습니다. [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) 및 [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html) 옵션에 대한 정보는 해당 API 참조를 확인하세요.

<span id="pass-options"></span>
## Sampler 옵션 설정
Sampler를 초기화할 때, 초기화 후, 또는 Sampler가 초기화된 후에 옵션을 업데이트할 수 있습니다. 이러한 기법을 사용하는 방법은 [옵션 소개](/guides/runtime-options-overview#options-precedence) 주제를 참조하세요.

또한 다음 섹션에서 설명하는 바와 같이 `run()` 메서드에서 `shots` 값을 설정할 수 있습니다.
<span id="run-method"></span>
### Run() 메서드
`run()`에 전달할 수 있는 값은 인터페이스에 정의된 것, 즉 `shots`뿐입니다. 이는 현재 실행에 대해 `default_shots`에 설정된 값을 덮어씁니다.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

<Admonition type="note">
Although shots specified in the PUB and in `run` have higher precedence, the job fails if `twirling` is enabled and the product of `num_randomizations` and `shots_per_randomization` is smaller than the `shots` value. In this scenario, `SamplerV2` is unable to allocate the shots among the specified `num_randomizations`.
</Admonition>

Example:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden
# if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### 특수한 경우
<span id="shots"></span>
#### Shots
`SamplerV2.run` 메서드는 두 가지 인수를 받습니다. PUB별 shots 값을 지정할 수 있는 PUB 목록과 shots 키워드 인수입니다. 이 shots 값들은 Sampler 실행 인터페이스의 일부이며, Runtime Sampler의 옵션과는 독립적입니다. Sampler 추상화를 준수하기 위해 옵션으로 지정된 값보다 우선합니다.

단, PUB나 run 키워드 인수에서 `shots`가 지정되지 않은 경우(또는 모두 `None`인 경우), 옵션의 shots 값이 사용되며, 특히 `default_shots`가 사용됩니다.

정리하면, 특정 PUB에 대해 Sampler에서 shots를 지정하는 우선순위는 다음과 같습니다.

1. PUB에서 shots를 지정하면 해당 값을 사용합니다.
2. `run`에서 `shots` 키워드 인수가 지정되면 해당 값을 사용합니다.
4. `twirling`이 활성화되어 있으면(기본값 True), [`twirling` 옵션](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)으로 지정된 `num_randomizations`와 `shots_per_randomization`의 곱이 사용됩니다.
5. `sampler.options.default_shots`가 지정되면 해당 값을 사용합니다.

따라서 가능한 모든 위치에서 shots가 지정된 경우, 가장 높은 우선순위를 가진 값(PUB에서 지정된 shots)이 사용됩니다.

예시: